## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-19-12-52-21 +0000,nyt,Live Updates: U.N. Court Rules Myanmar Committ...,https://www.nytimes.com/live/2026/11/19/world/...
1,2026-11-19-12-51-53 +0000,wapo,"At historic summit, Xi and Putin share global ...",https://www.washingtonpost.com/politics/2026/1...
2,2026-11-19-12-51-21 +0000,bbc,Badenoch defends amputee marathon runners clai...,https://www.bbc.co.uk/news/articles/cw62pz48vj...
3,2026-11-19-12-51-17 +0000,nyt,How the War in Iran Will Remake the Global Ene...,https://www.nytimes.com/2026/11/19/business/co...
4,2026-11-19-12-51-00 +0000,wsj,Oil Prices Retreat as Traders Weigh Supply Sur...,https://www.wsj.com/business/energy-oil/oil-pr...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2645/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
17,trump,57
107,anthropic,23
44,china,22
12,putin,21
220,new,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
5,2026-11-19-12-50-09 +0000,nyt,Live Updates: Trump Departs China After First-...,https://www.nytimes.com/live/2026/11/18/world/...,157
365,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...,155
160,2026-11-19-03-53-50 +0000,chitri,President Trump meets Xi and Putin in China wi...,https://www.chicagotribune.com/2026/11/18/trum...,153
90,2026-11-19-10-24-31 +0000,latimes,"'Making history': Trump, Putin and Xi meet in ...",https://www.latimes.com/politics/story/2026-11...,150
296,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,142


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
5,157,2026-11-19-12-50-09 +0000,nyt,Live Updates: Trump Departs China After First-...,https://www.nytimes.com/live/2026/11/18/world/...
188,94,2026-11-19-02-00-00 +0000,wsj,Jensen Huang’s ouster from Nvidia is giving ne...,https://www.wsj.com/politics/national-security...
296,85,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...
203,78,2026-11-19-01-11-00 +0000,wsj,Elon Musk said he felt “pretty foolish” about ...,https://www.wsj.com/politics/policy/elon-musk-...
278,68,2026-11-18-22-13-12 +0000,cbc,She says Russell Brand sexually assaulted her ...,https://www.cbc.ca/radio/asithappens/mia-latte...
424,62,2026-11-18-14-55-06 +0000,chitri,"Anthropic prices its IPO at $83 per share, val...",https://www.chicagotribune.com/2026/11/18/anth...
6,49,2026-11-19-12-50-00 +0000,wsj,World Court Finds Myanmar Committed Genocide A...,https://www.wsj.com/world/asia/world-court-fin...
69,49,2026-11-19-11-00-00 +0000,nypost,Amazon warns NYC customers they face price hik...,https://nypost.com/2026/11/19/business/in-new-...
246,48,2026-11-18-23-11-17 +0000,nypost,Iran and US trade tit-for-tat strikes as wild ...,https://nypost.com/2026/11/18/world-news/uae-o...
74,45,2026-11-19-11-00-00 +0000,latimes,"High tides, big storms: Next week will be key ...",https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
